<a href="https://colab.research.google.com/github/Nayab-khalid/FlyRank-AI-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nayab-khalid/FlyRank-AI-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
# ============================================================
# ML-09 — PART 1
# TWO PAPER FINDINGS + MY METHODOLOGY QUESTIONS
# ============================================================

# FINDING 1 — THE ANATOMY OF GROWING CONTENT
#
# The paper reports that content with rising impressions tends to
# be longer, younger, and slightly better positioned than content
# with falling impressions.
#
# The growing group has an average of about 3,180 words and
# 184 days of age, while the declining group has about 2,311 words
# and 230 days of age.
#
# The paper reports large groups of approximately 74,187 rising
# pages and 45,272 falling pages.
#
# The paper itself describes this as an observational comparison,
# so the result shows an association rather than proof that being
# younger or longer causes growth.
#
# METHODOLOGY QUESTION:
# Where exactly does the outcome label come from, and is the
# validation design strong enough to support the claim?
#
# In this finding, the groups are defined using rising versus
# falling impressions. The paper's metric description says trend
# direction is calculated from the change in impressions between
# the most recent 30 days and the previous 30 days.
#
# My question is whether the observed differences in age and
# word count could partly reflect differences in content history,
# client mix, or other confounding factors rather than the
# characteristics themselves causing better performance.
#
# A stronger validation design for a causal interpretation would
# compare similar pages or use a time-aware before-and-after design
# while controlling for important differences between pages.
#
# Therefore, I would treat this finding as measured and directional
# evidence about the observed portfolio, not as proof that increasing
# word count or reducing content age will automatically cause growth.


# FINDING 4 — THE FRESHNESS MULTIPLIER
#
# The paper reports that the 31-90 day freshness window is the
# strongest stable freshness band, with a growth-to-decline ratio
# of about 7.88:1.
#
# It also reports that 365+ day content refreshed within 30 days
# showed a 3.2x health increase, from 10.7 to 34.5, and 57x more
# impressions, from 71 to 4,039.
#
# The paper also warns that the 361+ freshness bucket is very
# unstable because it contains only one declining page in the
# local active-content sample.
#
# METHODOLOGY QUESTION:
# How is the refresh exposure defined relative to the outcome
# window, and does the validation design separate association
# from the effect of refreshing?
#
# The important question is whether pages that were refreshed were
# comparable to pages that were not refreshed before the outcome
# was measured. Older pages selected for refresh may already differ
# in demand, importance, quality, or historical performance.
#
# I would therefore want to know whether the comparison used a
# matched or otherwise controlled comparison group, and whether
# the impression and health measurements were taken after the
# refresh in a clearly separated outcome window.
#
# Without that stronger design, the result supports an observed
# association between recent refresh activity and stronger
# performance, but it does not by itself prove that the refresh
# caused the improvement.
#
# Overall, this is especially relevant to my own model because
# I should avoid turning freshness into a causal claim. I should
# use it as a measurable signal for decision-support and validate
# its usefulness with a leakage-safe, client-grouped or time-aware
# evaluation.


print("PART 1 - PAPER FINDINGS REVIEWED")
print("-" * 46)
print("Finding 1: anatomy of growing content (length, age, position)")
print("Finding 2: refresh/freshness logic")
print()
print("Label definition recorded from the paper's metric description:")
print("  trend direction = change in impressions between the most recent")
print("  30 days and the previous 30 days.")
print()
print("NOTE ADDED LATER: I wrote that definition down here in Week 6 while")
print("reviewing someone else's finding, and did not check it against my")
print("own feature list. Part 3 of this notebook is where that check runs.")


PART 1 - PAPER FINDINGS REVIEWED
----------------------------------------------
Finding 1: anatomy of growing content (length, age, position)
Finding 2: refresh/freshness logic

Label definition recorded from the paper's metric description:
  trend direction = change in impressions between the most recent
  30 days and the previous 30 days.

NOTE ADDED LATER: I wrote that definition down here in Week 6 while
reviewing someone else's finding, and did not check it against my
own feature list. Part 3 of this notebook is where that check runs.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# ============================================================
# ML-09 — PART 2
# MY MODEL UNDER AN HONEST SPLIT
# ============================================================

# REASONING:
# This audit answers two questions at once.
#
# 1. Does performance change when client overlap is removed?
#    A row-based split lets pages from the same client sit on both
#    sides. A client-grouped split does not.
#
# 2. How much of my Week-5 result came from target leakage?
#    My original 26-feature list contained the two columns the
#    decline label is computed from. Part 3 of this notebook shows
#    the check that found it. Here I report both feature sets so the
#    two effects can be separated.

%pip install -q pandas numpy scikit-learn

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

possible_paths = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "/content/data/raw/content_refresh_anonymized.csv"
]

csv_path = next(
    (p for p in possible_paths if os.path.exists(p)),
    None
)

if csv_path is None:
    csv_url = (
        "https://raw.githubusercontent.com/"
        "Nayab-khalid/FlyRank-AI-Internship/"
        "main/data/raw/content_refresh_anonymized.csv"
    )
    df = pd.read_csv(csv_url)
else:
    df = pd.read_csv(csv_path)

# ------------------------------------------------------------
# LABEL
# ------------------------------------------------------------

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# ------------------------------------------------------------
# TWO FEATURE SETS
# ------------------------------------------------------------

numeric_full = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

# The label's own ingredients, plus the same windows on clicks and sessions.
RECENT_WINDOW = [
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d"
]

numeric_clean = [
    c for c in numeric_full
    if c not in RECENT_WINDOW
]

categorical_features = [
    "content_type",
    "main_intent",
    "competition_level"
]

y = df["is_declining_label"]
groups = df["client_id"]

# ------------------------------------------------------------
# MODEL PIPELINE
# ------------------------------------------------------------

def make_model(numeric_features):
    preprocessor = ColumnTransformer([
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ))
            ]),
            categorical_features
        )
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

# ------------------------------------------------------------
# EVALUATE ONE FEATURE SET UNDER ONE SPLIT
# ------------------------------------------------------------

def evaluate(numeric_features, split):

    X = df[numeric_features + categorical_features]

    if split == "row":
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.20,
            random_state=42,
            stratify=y
        )
    else:
        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=0.20,
            random_state=42
        )
        train_idx, test_idx = next(
            splitter.split(X, y, groups=groups)
        )
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

    model = make_model(numeric_features)
    model.fit(X_train, y_train)

    probability = model.predict_proba(X_test)[:, 1]

    return average_precision_score(y_test, probability)

# ------------------------------------------------------------
# BEFORE / AFTER TABLE
# ------------------------------------------------------------

rows = []

for set_name, numeric_features in (
    ("26 features (original)", numeric_full),
    ("20 features (leak removed)", numeric_clean)
):
    for split_name, split in (
        ("Row-based split", "row"),
        ("Client-grouped split", "group")
    ):
        rows.append({
            "feature_set": set_name,
            "validation_design": split_name,
            "average_precision": evaluate(numeric_features, split)
        })

comparison = pd.DataFrame(rows)

print("=" * 60)
print("BEFORE / AFTER VALIDATION")
print("=" * 60)

display(comparison)

grouped_full = comparison.query(
    "feature_set == '26 features (original)' "
    "and validation_design == 'Client-grouped split'"
)["average_precision"].iloc[0]

grouped_clean = comparison.query(
    "feature_set == '20 features (leak removed)' "
    "and validation_design == 'Client-grouped split'"
)["average_precision"].iloc[0]

row_clean = comparison.query(
    "feature_set == '20 features (leak removed)' "
    "and validation_design == 'Row-based split'"
)["average_precision"].iloc[0]

print("\nCost of removing client overlap (corrected features):",
      round(row_clean - grouped_clean, 6))

print("Cost of removing the leaked features (grouped split):",
      round(grouped_full - grouped_clean, 6))

print(
    "\nThe client-grouped result on the corrected feature set is the "
    "number I report. It is the most conservative of the four."
)


Note: you may need to restart the kernel to use updated packages.


BEFORE / AFTER VALIDATION


,feature_set,validation_design,average_precision
0,26 features (original),Row-based split,0.935522
1,26 features (original),Client-grouped split,0.871535
2,20 features (leak removed),Row-based split,0.693227
3,20 features (leak removed),Client-grouped split,0.596008



Cost of removing client overlap (corrected features): 0.09722
Cost of removing the leaked features (grouped split): 0.275527

The client-grouped result on the corrected feature set is the number I report. It is the most conservative of the four.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# ============================================================
# ML-09 — PART 3
# LEAKAGE AUDIT
# ============================================================

# REASONING:
# My earlier audits checked whether forbidden columns appeared in the
# feature list BY NAME. That check cannot detect a label that has been
# rebuilt from columns which are individually allowed, so this audit
# adds two tests that can.
#
# Test A — the formula test.
# docs/data-dictionary.md documents exactly how the label is built.
# I apply that formula to my own feature set and measure agreement.
#
# Test B — the reconstruction probe.
# I train a random forest on the features alone. If a probe reaches
# near-perfect accuracy on a task where my transparent rule baseline
# sits near chance, the features contain the answer rather than
# evidence for it.
#
# Test C — the name check I already had, kept as a floor.

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# ------------------------------------------------------------
# FINAL FEATURE LIST (after the Part 2 correction)
# ------------------------------------------------------------

final_features = numeric_clean + categorical_features

original_features = numeric_full + categorical_features

print("=" * 60)
print("LEAKAGE AUDIT")
print("=" * 60)

print("Original feature count:", len(original_features))
print("Final feature count:", len(final_features))

# ------------------------------------------------------------
# TEST C — NAME CHECK
# ------------------------------------------------------------

forbidden = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id",
    "content_id"
]

leakage_found = [
    feature
    for feature in final_features
    if feature in forbidden
]

print("\nTEST C - NAME CHECK")
print("Forbidden fields found by name:", len(leakage_found))
print("Result:", "PASS" if len(leakage_found) == 0 else "FAIL")
print("Note: this test also passed while the label was fully rebuildable.")

# ------------------------------------------------------------
# TEST A — FORMULA TEST
# ------------------------------------------------------------

prev_30d = pd.to_numeric(df["impressions_prev_30d"], errors="coerce")
last_30d = pd.to_numeric(df["impressions_last_30d"], errors="coerce")

documented_rule = (
    ((last_30d - prev_30d) / prev_30d.replace(0, np.nan)) * 100 < -20
).fillna(False).astype(int)

agreement = (documented_rule == df["is_declining_label"]).mean()

ingredients = ["impressions_last_30d", "impressions_prev_30d"]

still_present = [
    c for c in ingredients
    if c in final_features
]

print("\nTEST A - FORMULA TEST")
print("Documented rule vs label agreement:", round(agreement, 6))
print("Label ingredients still in the feature list:", still_present)
print("Result:", "PASS" if len(still_present) == 0 else "FAIL")

# ------------------------------------------------------------
# TEST B — RECONSTRUCTION PROBE
# ------------------------------------------------------------

def reconstruction_accuracy(numeric_features):
    probe = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
    return cross_val_score(
        probe,
        df[numeric_features].fillna(0),
        df["is_declining_label"],
        cv=3,
        scoring="accuracy"
    ).mean()

probe_before = reconstruction_accuracy(numeric_full)
probe_after = reconstruction_accuracy(numeric_clean)
base_rate = df["is_declining_label"].mean()

print("\nTEST B - RECONSTRUCTION PROBE")
print("Base rate (majority class):", round(base_rate, 4))
print("Probe accuracy, 23 original numeric features:", round(probe_before, 4))
print("Probe accuracy, 17 corrected numeric features:", round(probe_after, 4))
print("Result:", "PASS" if probe_after < 0.90 else "FAIL")

print("""
LEAKAGE CONCLUSION:

The name check alone certified a feature set from which the label
could be rebuilt exactly. The formula test found it: the decline
label is defined from impressions_last_30d and impressions_prev_30d,
and both were being used as model features.

Those two columns, and the same two windows measured on clicks and
sessions, are now excluded. The formula test and the reconstruction
probe both pass on the corrected list.

One risk remains open and is stated in the paper. The surviving 90-day
aggregates cover a window that contains the 30 days the label is
measured over, so these features are contemporaneous with the outcome
rather than prior to it. This is an association study, not a forecast.
Any future-window variable must be excluded if it overlaps the outcome
period.
""")


LEAKAGE AUDIT
Original feature count: 26
Final feature count: 20

TEST C - NAME CHECK
Forbidden fields found by name: 0
Result: PASS
Note: this test also passed while the label was fully rebuildable.

TEST A - FORMULA TEST
Documented rule vs label agreement: 1.0
Label ingredients still in the feature list: []
Result: PASS



TEST B - RECONSTRUCTION PROBE
Base rate (majority class): 0.5421
Probe accuracy, 23 original numeric features: 0.9122
Probe accuracy, 17 corrected numeric features: 0.6994
Result: PASS

LEAKAGE CONCLUSION:

The name check alone certified a feature set from which the label
could be rebuilt exactly. The formula test found it: the decline
label is defined from impressions_last_30d and impressions_prev_30d,
and both were being used as model features.

Those two columns, and the same two windows measured on clicks and
sessions, are now excluded. The formula test and the reconstruction
probe both pass on the corrected list.

One risk remains open and is stated in the paper. The surviving 90-day
aggregates cover a window that contains the 30 days the label is
measured over, so these features are contemporaneous with the outcome
rather than prior to it. This is an association study, not a forecast.
Any future-window variable must be excluded if it overlaps the outcome
period.



## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [4]:
# ============================================================
# ML-09 — PART 4
# CLAIM REWRITE
# ============================================================

# ORIGINAL BOLD CLAIM:
# "The Logistic Regression model predicts which pages will decline
# and can identify pages that Google will rank lower."

# SAFE REWRITE:
# "In this dataset, the Logistic Regression model showed measured
# and directional ability to rank pages associated with the
# observed decline label. Its performance should be interpreted
# as decision-support evidence rather than a prediction of
# Google's ranking algorithm or a causal claim about why a page
# declined."

print("""
ORIGINAL CLAIM:
The Logistic Regression model predicts which pages will decline
and can identify pages that Google will rank lower.

REWRITTEN CLAIM:
In this dataset, the Logistic Regression model showed measured
and directional ability to rank pages associated with the observed
decline label. Its performance should be interpreted as
decision-support evidence rather than a prediction of Google's
ranking algorithm or a causal claim about why a page declined.
""")



ORIGINAL CLAIM:
The Logistic Regression model predicts which pages will decline
and can identify pages that Google will rank lower.

REWRITTEN CLAIM:
In this dataset, the Logistic Regression model showed measured
and directional ability to rank pages associated with the observed
decline label. Its performance should be interpreted as
decision-support evidence rather than a prediction of Google's
ranking algorithm or a causal claim about why a page declined.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.